# 02 IT Jobs EDA and Cleaning

This notebook is used for the initial exploration and cleaning of the LinkedIn Job Postings Dataset.

Main goals:
- load the raw job postings dataset
- inspect dataset structure
- check missing values and duplicates
- identify useful columns
- remove invalid job postings
- filter job postings related to the IT domain
- clean job descriptions
- save cleaned IT job postings for later structured extraction and market analysis

## 1. Imports and Settings

In [1]:
import pandas as pd
import numpy as np
import re
import html

from pathlib import Path

In [2]:
RAW_JOB_DIR = Path("../data/raw/jobs")
PROCESSED_JOBS_DIR = Path("../data/processed/jobs")

csv_file='postings.csv'

In [3]:
job_path=RAW_JOB_DIR/csv_file
print(job_path)

..\data\raw\jobs\postings.csv


## 2. Load Small Sample First

In [4]:
df = pd.read_csv(job_path)

In [5]:
df.shape

(123849, 31)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  str    
 2   title                       123849 non-null  str    
 3   description                 123842 non-null  str    
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   str    
 6   location                    123849 non-null  str    
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  str    
 12  applies                     23320 non-null   float64
 13  original_listed_time     

In [7]:
df.head(10)

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0
5,91700727,Downtown Raleigh Alliance,Economic Development and Planning Intern,Job summary:The Economic Development & Plannin...,20.0,HOURLY,"Raleigh, NC",1481176.0,9.0,NaN,...,NaN,1.713456e+12,NaN,0,INTERNSHIP,USD,BASE_SALARY,35360.0,27601.0,37183.0
6,103254301,Raw Cereal,Producer,Company DescriptionRaw Cereal is a creative de...,300000.0,YEARLY,United States,81942316.0,7.0,NaN,...,NaN,1.712861e+12,NaN,0,CONTRACT,USD,BASE_SALARY,180000.0,NaN,NaN
7,112576855,NaN,Building Engineer,Summary: Due to the pending retirement of our ...,120000.0,YEARLY,"San Francisco, CA",NaN,2.0,NaN,...,NaN,1.712443e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,105000.0,94101.0,6075.0
8,1218575,Children's Nebraska,Respiratory Therapist,"At Children’s, the region’s only full-service ...",NaN,NaN,"Omaha, NE",721189.0,3.0,NaN,...,• Requires the ability to communicate effectiv...,1.712348e+12,NaN,0,FULL_TIME,NaN,NaN,NaN,68102.0,31055.0
9,2264355,Bay West Church,Worship Leader,It is an exciting time to be a part of our chu...,NaN,MONTHLY,"Palm Bay, FL",28631247.0,5.0,350.0,...,"Knowledge, Skills and Abilities: 1. Proficient...",1.712456e+12,NaN,0,PART_TIME,USD,BASE_SALARY,4200.0,32905.0,12009.0


## 3. Select Useful Columns

In [8]:
df["currency"].nunique()
df["currency"].unique().tolist()

['USD', nan, 'CAD', 'BBD', 'EUR', 'AUD', 'GBP']

In [9]:
selected_columns = [
    "title",
    "description",
    "skills_desc",
    "company_name",
    "formatted_work_type",
    "work_type",
    "remote_allowed",
    "formatted_experience_level",
    "listed_time",
    "original_listed_time",
    "expiry",
    "job_posting_url",
    "min_salary",
    "max_salary",
    "med_salary",
    "currency",
    "compensation_type",
    "pay_period",
    "normalized_salary",
    "views",
    "applies"
]


In [10]:
df_jobs = df[selected_columns]
df_jobs

,title,description,skills_desc,company_name,formatted_work_type,work_type,remote_allowed,formatted_experience_level,listed_time,original_listed_time,...,job_posting_url,min_salary,max_salary,med_salary,currency,compensation_type,pay_period,normalized_salary,views,applies
0,Marketing Coordinator,Job descriptionA leading real estate firm in N...,Requirements: \n\nWe are seeking a College or ...,Corcoran Sawyer Smith,Full-time,FULL_TIME,NaN,NaN,1.713398e+12,1.713398e+12,...,https://www.linkedin.com/jobs/view/921716/?trk...,17.0,20.0,NaN,USD,BASE_SALARY,HOURLY,38480.0,20.0,2.0
1,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",NaN,NaN,Full-time,FULL_TIME,NaN,NaN,1.712858e+12,1.712858e+12,...,https://www.linkedin.com/jobs/view/1829192/?tr...,30.0,50.0,NaN,USD,BASE_SALARY,HOURLY,83200.0,1.0,NaN
2,Assitant Restaurant Manager,The National Exemplar is accepting application...,We are currently accepting resumes for FOH - A...,The National Exemplar,Full-time,FULL_TIME,NaN,NaN,1.713278e+12,1.713278e+12,...,https://www.linkedin.com/jobs/view/10998357/?t...,45000.0,65000.0,NaN,USD,BASE_SALARY,YEARLY,55000.0,8.0,NaN
3,Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,This position requires a baseline understandin...,"Abrams Fensterman, LLP",Full-time,FULL_TIME,NaN,NaN,1.712896e+12,1.712896e+12,...,https://www.linkedin.com/jobs/view/23221523/?t...,140000.0,175000.0,NaN,USD,BASE_SALARY,YEARLY,157500.0,16.0,NaN
4,Service Technician,Looking for HVAC service tech with experience ...,NaN,NaN,Full-time,FULL_TIME,NaN,NaN,1.713452e+12,1.713452e+12,...,https://www.linkedin.com/jobs/view/35982263/?t...,60000.0,80000.0,NaN,USD,BASE_SALARY,YEARLY,70000.0,3.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123844,Title IX/Investigations Attorney,Our Walnut Creek office is currently seeking a...,NaN,Lozano Smith,Full-time,FULL_TIME,NaN,Mid-Senior level,1.713571e+12,1.713571e+12,...,https://www.linkedin.com/jobs/view/3906267117/...,120000.0,195000.0,NaN,USD,BASE_SALARY,YEARLY,157500.0,1.0,NaN
123845,"Staff Software Engineer, ML Serving Platform",About Pinterest:\n\nMillions of people across ...,NaN,Pinterest,Full-time,FULL_TIME,1.0,Mid-Senior level,1.713572e+12,1.713572e+12,...,https://www.linkedin.com/jobs/view/3906267126/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN
123846,"Account Executive, Oregon/Washington",Company Overview\n\nEPS Learning is a leading ...,NaN,EPS Learning,Full-time,FULL_TIME,1.0,Mid-Senior level,1.713572e+12,1.713572e+12,...,https://www.linkedin.com/jobs/view/3906267131/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN
123847,Business Development Manager,The Business Development Manager is a 'hunter'...,NaN,Trelleborg Applied Technologies,Full-time,FULL_TIME,1.0,NaN,1.713573e+12,1.713573e+12,...,https://www.linkedin.com/jobs/view/3906267195/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN


## 4. Missing Values Check

In [11]:
missing_values=df_jobs.isna().sum().sort_values(ascending=False)

In [12]:
missing_values

skills_desc                   121410
med_salary                    117569
remote_allowed                108603
applies                       100529
min_salary                     94056
max_salary                     94056
currency                       87776
pay_period                     87776
compensation_type              87776
normalized_salary              87776
formatted_experience_level     29409
company_name                    1719
views                           1689
description                        7
title                              0
formatted_work_type                0
job_posting_url                    0
work_type                          0
expiry                             0
original_listed_time               0
listed_time                        0
dtype: int64

In [13]:
missing_percentage = (df_jobs.isna().mean() * 100).sort_values(ascending=False)

missing_percentage

skills_desc                   98.030666
med_salary                    94.929309
remote_allowed                87.689848
applies                       81.170619
min_salary                    75.944093
max_salary                    75.944093
currency                      70.873402
pay_period                    70.873402
compensation_type             70.873402
normalized_salary             70.873402
formatted_experience_level    23.745852
company_name                   1.387981
views                          1.363757
description                    0.005652
title                          0.000000
formatted_work_type            0.000000
job_posting_url                0.000000
work_type                      0.000000
expiry                         0.000000
original_listed_time           0.000000
listed_time                    0.000000
dtype: float64

## 5.Handle Missing Values

In [14]:
df_jobs = df_jobs.dropna(subset=["title", "description"]).copy()

In [15]:
optional_unknown_columns = [
    "company_name",
    "formatted_experience_level",
]

for col in optional_unknown_columns:
    if col in df_jobs.columns:
        df_jobs[col] = df_jobs[col].fillna("Unknown")

In [16]:
columns_for_drop=[
    "min_salary",
    "max_salary",
    "med_salary",
    "currency",
    "pay_period",
    "compensation_type",
    "normalized_salary",
    "skills_desc",
    "remote_allowed",
    "applies",
    "views"
]

In [17]:
df_jobs = df_jobs.drop(columns=columns_for_drop).copy()

In [18]:
missing_percentage = (df_jobs.isna().mean() * 100).sort_values(ascending=False)

missing_percentage

title                         0.0
description                   0.0
company_name                  0.0
formatted_work_type           0.0
work_type                     0.0
formatted_experience_level    0.0
listed_time                   0.0
original_listed_time          0.0
expiry                        0.0
job_posting_url               0.0
dtype: float64

## 6. Duplicate Rows Check

In [19]:
full_duplicates = df_jobs.duplicated().sum()

In [20]:
full_duplicates

np.int64(0)

In [21]:
title_description_duplicates = df_jobs.duplicated(
    subset=["title", "description"]
).sum()

title_description_duplicates

np.int64(12944)

In [22]:
rows_before = len(df_jobs)

df_jobs = df_jobs.drop_duplicates(
    subset=["title", "description"]
).reset_index(drop=True)

## 7. Clean Job Text


In [23]:
def clean_text(text):

    if pd.isna(text):
        return ""
    
    text = str(text)
    text = html.unescape(text)
    text = re.sub(r"<.*?>", " ", text)
    text = text.replace("\t", " ")
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")

    text = re.sub(r"\s+", " ", text)

    text = text.strip()
    
    return text

In [24]:
df_jobs["clean_job_title"] = df_jobs["title"].apply(clean_text)
df_jobs["clean_job_description"] = df_jobs["description"].apply(clean_text)


In [25]:
print(df_jobs.iloc[0][ "clean_job_description"])

Job descriptionA leading real estate firm in New Jersey is seeking an administrative Marketing Coordinator with some experience in graphic design. You will be working closely with our fun, kind, ambitious members of the sales team and our dynamic executive team on a daily basis. This is an opportunity to be part of a fast-growing, highly respected real estate brokerage with a reputation for exceptional marketing and extraordinary culture of cooperation and inclusion.Who you are:You must be a well-organized, creative, proactive, positive, and most importantly, kind-hearted person. Please, be responsible, respectful, and cool-under-pressure. Please, be proficient in Adobe Creative Cloud (Indesign, Illustrator, Photoshop) and Microsoft Office Suite. Above all, have fantastic taste and be a good-hearted, fun-loving person who loves working with people and is eager to learn.Role:Our office is a fast-paced environment. You’ll work directly with a Marketing team and communicate daily with oth

In [26]:
print(df_jobs.iloc[0][ "description"])

Job descriptionA leading real estate firm in New Jersey is seeking an administrative Marketing Coordinator with some experience in graphic design. You will be working closely with our fun, kind, ambitious members of the sales team and our dynamic executive team on a daily basis. This is an opportunity to be part of a fast-growing, highly respected real estate brokerage with a reputation for exceptional marketing and extraordinary culture of cooperation and inclusion.Who you are:You must be a well-organized, creative, proactive, positive, and most importantly, kind-hearted person. Please, be responsible, respectful, and cool-under-pressure. Please, be proficient in Adobe Creative Cloud (Indesign, Illustrator, Photoshop) and Microsoft Office Suite. Above all, have fantastic taste and be a good-hearted, fun-loving person who loves working with people and is eager to learn.Role:Our office is a fast-paced environment. You’ll work directly with a Marketing team and communicate daily with oth

## 8. Check Job Description Length

In [27]:
df_jobs["clean_job_description"].str.len().describe()

count    110898.000000
mean       3743.418682
std        2163.909355
min           2.000000
25%        2138.000000
50%        3424.000000
75%        4967.000000
max       23106.000000
Name: clean_job_description, dtype: float64

In [28]:
min_description_words = 250

df_jobs = df_jobs[
    df_jobs["clean_job_description"].fillna("").str.split().str.len() >= min_description_words
].copy()


## 9. Create Combined Text Column for IT Filtering

In [29]:
df_jobs["combined_text"] = (
    df_jobs["clean_job_title"].fillna("") + " " +
    df_jobs["clean_job_description"].fillna("")
).str.lower()

df_jobs[["clean_job_title", "combined_text"]].head()

,clean_job_title,combined_text
0,Marketing Coordinator,marketing coordinator job descriptiona leading...
1,Mental Health Therapist/Counselor,mental health therapist/counselor at aspen the...
5,Economic Development and Planning Intern,economic development and planning intern job s...
7,Building Engineer,building engineer summary: due to the pending ...
8,Respiratory Therapist,"respiratory therapist at children’s, the regio..."


## 10. Filter IT-Related Job Postings

In [30]:
it_title_keywords = [
    "software developer",
    "software engineer",
    "frontend developer",
    "front-end developer",
    "backend developer",
    "back-end developer",
    "full stack developer",
    "fullstack developer",
    "web developer",
    "mobile developer",
    "android developer",
    "ios developer",
    "data analyst",
    "data scientist",
    "machine learning engineer",
    "ml engineer",
    "ai engineer",
    "devops engineer",
    "cloud engineer",
    "qa engineer",
    "quality assurance engineer",
    "software tester",
    "automation tester",
    "cybersecurity analyst",
    "security analyst",
    "system administrator",
    "database administrator",
    "bi analyst",
    "business intelligence analyst",
    "it support",
    "network administrator"
]

In [31]:
it_tech_keywords = [
    "python",
    "java",
    "javascript",
    "typescript",
    "c#",
    "c++",
    "php",
    "ruby",
    "go",
    "kotlin",
    "swift",
    "sql",
    "mysql",
    "postgresql",
    "mongodb",
    "oracle",
    "react",
    "angular",
    "vue",
    "node.js",
    "node",
    "django",
    "flask",
    "spring boot",
    ".net",
    "asp.net",
    "docker",
    "kubernetes",
    "jenkins",
    "git",
    "github",
    "gitlab",
    "aws",
    "azure",
    "google cloud",
    "gcp",
    "linux",
    "rest api",
    "api",
    "power bi",
    "tableau",
    "pandas",
    "numpy",
    "spark",
    "tensorflow",
    "pytorch",
    "selenium",
    "jira",
    "ci/cd",
    "devops",
    "cloud",
    "cybersecurity",
    "machine learning",
    "data analysis",
    "data visualization"
]

In [32]:
def contains_any_keyword(text, keywords):
    text = str(text).lower()
    return any(keyword.lower() in text for keyword in keywords)


def count_keywords(text, keywords):
    text = str(text).lower()
    return sum(keyword.lower() in text for keyword in keywords)

In [33]:
df_jobs["title_has_it_keyword"] = df_jobs["clean_job_title"].apply(
    lambda x: contains_any_keyword(x, it_title_keywords)
)

df_jobs["tech_keyword_count"] = df_jobs["combined_text"].apply(
    lambda x: count_keywords(x, it_tech_keywords)
)

df_jobs[["clean_job_title", "title_has_it_keyword", "tech_keyword_count"]].head(20)

,clean_job_title,title_has_it_keyword,tech_keyword_count
0,Marketing Coordinator,False,2
1,Mental Health Therapist/Counselor,False,2
5,Economic Development and Planning Intern,False,4
7,Building Engineer,False,1
8,Respiratory Therapist,False,2
10,Inside Customer Service Associate,False,0
11,Project Architect,False,2
12,Appalachian Highlands Women's Business Center,False,2
14,Senior Product Marketing Manager,False,1
15,Osteogenic Loading Coach,False,2


### 11. IT Filtering Logic

In [34]:
df_jobs_it = df_jobs[
    (df_jobs["title_has_it_keyword"] == True) |
    (df_jobs["tech_keyword_count"] >= 5)
].copy()


In [35]:
df_jobs_it.shape

(6495, 15)

In [36]:
df_jobs_it[[
    "clean_job_title",
    "clean_job_description",
    "tech_keyword_count"
]].head(10)

,clean_job_title,clean_job_description,tech_keyword_count
78,Full Stack Engineer,"Location: Remote Company Overview:SkillFit, a ...",17
146,Senior Developer,This individual will work with a high performa...,9
181,Senior Software Engineer,"StyleAI is the AI-powered, all-in-one unified ...",10
196,DDI Engineer,Title: Infoblox/DNS EngineerLocation: 6860 Yos...,6
251,Cybersecurity Test Engineer – Remote,Decision Point Security Inc. is currently seek...,6
266,Principal Backend Engineer,Principal Backend Engineer - Join HireBus and ...,9
284,Unpaid Internship: Development Team,Bright Sparks Academy Internship Program Befor...,7
354,Engineers / Marketing / Various,"General Role: We are hiring for engineers, mar...",6
360,Cloud Platform/ Big Data Engineer,About Subaru Research and Development:Do you c...,12
366,Data Science Software Engineer,Company Overview Sovrinti is a full-service en...,17


## 12. Basic IT Job Category Classification

In [37]:
#The job categories and their associated keywords were derived from the O*NET website

IT_CATEGORY_KEYWORDS = {
    "Data Analytics": [
        "data analyst", "business intelligence", "bi analyst", "power bi", "tableau",
        "excel", "dashboard", "reporting", "looker", "sap", "snowflake", "sas", "sql"
    ],

    "Data Science / AI": [
        "data scientist", "machine learning", "deep learning", "ml engineer", "ai engineer",
        "artificial intelligence", "python", "r", "tensorflow", "pytorch", "spark",
        "hadoop", "nlp", "data mining", "data modeling"
    ],

    "Data Engineering": [
        "data engineer", "database developer", "database architect", "database administrator",
        "dba", "etl", "data warehouse", "data pipeline", "big data", "sql",
        "postgresql", "mysql", "oracle", "snowflake", "spark", "hadoop", "airflow", "kafka"
    ],

    "Frontend Development": [
        "frontend", "front-end", "web developer", "javascript", "typescript",
        "react", "angular", "vue", "html", "css", "scss", "responsive design"
    ],

    "Backend Development": [
        "backend", "back-end", "software developer", "software engineer", "java",
        "python", "sql", "api", "rest api", "node.js", "c#", "c++",
        ".net", "django", "flask", "spring", "spring boot"
    ],

    "Full Stack Development": [
        "full stack", "fullstack", "full-stack", "mern", "mean stack",
        "react", "node.js", "javascript", "typescript", "api", "backend", "frontend"
    ],

    "DevOps / Cloud": [
        "devops", "cloud engineer", "aws", "azure", "gcp", "kubernetes",
        "docker", "jenkins", "ci/cd", "linux", "git", "github",
        "terraform", "ansible", "bash", "powershell"
    ],

    "QA / Testing": [
        "qa", "quality assurance", "tester", "software test engineer", "test engineer",
        "selenium", "jira", "junit", "automation testing", "test automation",
        "manual testing", "jenkins", "bug tracking"
    ],

    "Cybersecurity": [
        "cybersecurity", "information security", "security analyst", "security engineer",
        "penetration testing", "vulnerability", "siem", "soc", "splunk",
        "active directory", "linux", "powershell", "bash", "terraform", "docker"
    ],

    "IT Support / Administration": [
        "system administrator", "systems administrator", "network administrator", "it support",
        "helpdesk", "technical support", "linux", "windows server", "active directory",
        "powershell", "servicenow", "aws", "azure", "bash"
    ],

    "Systems / IT Analysis": [
        "systems analyst", "system analyst", "business systems analyst", "it analyst",
        "applications analyst", "programmer analyst", "requirements analysis",
        "business requirements", "functional requirements", "system requirements",
        "process analysis", "uml", "use case", "workflow", "documentation"
    ]
}

In [38]:
CATEGORY_PRIORITY = [
    "Full Stack Development", "Data Science / AI", "Database / Data Engineering",
    "Data Analytics", "Backend Development", "Frontend Development",
    "DevOps / Cloud", "QA / Testing", "Cybersecurity",
    "Systems / IT Analysis", "IT Support / Administration"
]

In [39]:
category_priority_map = {
    category: i for i, category in enumerate(CATEGORY_PRIORITY)
}

In [40]:
def keyword_found(text, keyword):
    keyword = keyword.lower()

    if any(char in keyword for char in [".", "+", "#", "/"]):
        return re.search(re.escape(keyword), text) is not None

    pattern = rf"(?<![a-z0-9]){re.escape(keyword)}(?![a-z0-9])"
    return re.search(pattern, text) is not None

In [41]:
def classify_it_job_multilabel(text):
    text = "" if pd.isna(text) else str(text).lower()

    category_scores = {}
    matched_keywords = {}

    for category, keywords in IT_CATEGORY_KEYWORDS.items():
        matches = []

        for keyword in keywords:
            if keyword_found(text, keyword):
                matches.append(keyword)

        if matches:
            category_scores[category] = len(matches)
            matched_keywords[category] = matches

    if not category_scores:
        return pd.Series({
            "primary_it_job_category": "Uncategorized",
            "all_it_job_categories": "Uncategorized",
            "category_scores": {},
            "matched_keywords": {},
            "is_multi_category": False
        })

    sorted_categories = sorted(
        category_scores.keys(),
        key=lambda cat: (
            -category_scores[cat],
            category_priority_map.get(cat, 999)
        )
    )

    return pd.Series({
        "primary_it_job_category": sorted_categories[0],
        "all_it_job_categories": "; ".join(sorted_categories),
        "category_scores": category_scores,
        "matched_keywords": matched_keywords,
        "is_multi_category": len(sorted_categories) > 1
    })

In [42]:
# classification_cols = [
#     "primary_it_job_category",
#     "all_it_job_categories",
#     "category_scores",
#     "matched_keywords",
#     "is_multi_category"
# ]

# df_jobs_it = df_jobs_it.drop(columns=classification_cols, errors="ignore")

In [43]:
classification_results = df_jobs_it["combined_text"].apply(classify_it_job_multilabel)

In [44]:
df_jobs_it = pd.concat([df_jobs_it, classification_results], axis=1)

In [45]:
df_jobs_it["primary_it_job_category"].value_counts()

primary_it_job_category
Backend Development            1520
DevOps / Cloud                 1324
Data Analytics                 1027
Data Science / AI               789
Full Stack Development          386
Data Engineering                383
Frontend Development            349
IT Support / Administration     250
Cybersecurity                   239
QA / Testing                    116
Systems / IT Analysis            83
Uncategorized                    29
Name: count, dtype: int64

## 13. Save Cleaned IT Job Postings Dataset


In [46]:
cleaned_jobs_path = PROCESSED_JOBS_DIR / "cleaned_it_job_postings.csv"

df_jobs_it.to_csv(cleaned_jobs_path, index=False)

print(f"Cleaned IT job postings saved to: {cleaned_jobs_path}")

Cleaned IT job postings saved to: ..\data\processed\jobs\cleaned_it_job_postings.csv
